### B1 Pipeline : Technical Indicators Only

In [1]:
import polars as pl
import numpy as np
import os
import joblib
import pickle
from datetime import date

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score,
)

In [10]:
# Config
Model_path = r"C:\Users\satya\OneDrive\Documents\Work\Post-Earnings-Forecast\src\modeling\data\tech_modeling_table.parquet"

Train_window = 7
Folds = [
    (date(y - Train_window, 1, 1), date(y, 1, 1), date(y + 1, 1, 1))
    for y in range(2021, 2026)
]

CLF_MODELS = [
    ("Logistic Regression", LogisticRegression(max_iter=1000, class_weight="balanced"), True),
    ("Random Forest",       RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1), False),
    ("XGBoost",             XGBClassifier(n_estimators=200, eval_metric="logloss", random_state=42), False),
]

REG_MODELS = [
    ("Linear Regression", LinearRegression(), True),
    ("Random Forest",     RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1), False),
    ("XGBoost",           XGBRegressor(n_estimators=200, random_state=42), False),
]

In [11]:
# Load precomputed table
df_model = pl.read_parquet(Model_path)
print(f"Modeling table: {df_model.shape}")
print(f"Target: {df_model['target_direction'].mean():.1%} positive")

feature_cols = [c for c in df_model.columns
                if c not in ["symbol", "earnings_date", "target_return", "target_direction"]]

Modeling table: (24048, 220)
Target: 56.3% positive


In [14]:
# Walk-forward validation
clf_results = {name: {"fold_acc": [], "preds": [], "probs": [], "true": []}
               for name, _, _ in CLF_MODELS}
clf_results["B0 Random"] = {"fold_acc": [], "preds": [], "probs": [], "true": []}

reg_results = {name: {"fold_mae": [], "fold_rmse": [], "preds": [], "true": []}
               for name, _, _ in REG_MODELS}
reg_results["B0 Mean"] = {"fold_mae": [], "fold_rmse": [], "preds": [], "true": []}

np.random.seed(42)

print(f"Walk-forward: {len(Folds)} folds (sliding {Train_window}-year window)")
print("-" * 60)

for fold_num, (train_start, test_start, test_end) in enumerate(Folds, 1):
    train = df_model.filter(
        (pl.col("earnings_date") >= train_start) & (pl.col("earnings_date") < test_start)
    )
    test = df_model.filter(
        (pl.col("earnings_date") >= test_start) & (pl.col("earnings_date") < test_end)
    )

    if test.is_empty():
        print(f"Fold {fold_num}: no test data, skipping")
        continue

    print(f"\nFold {fold_num}: train [{train_start.year}-{test_start.year - 1}] "
          f"({train.shape[0]:,}) -> test [{test_start.year}] ({test.shape[0]:,})")

    X_train = train.select(feature_cols).to_numpy()
    X_test = test.select(feature_cols).to_numpy()
    y_train_dir = train["target_direction"].to_numpy()
    y_test_dir = test["target_direction"].to_numpy()
    y_train_ret = train["target_return"].to_numpy()
    y_test_ret = test["target_return"].to_numpy()

    # preprocessing (fitted on this fold's train only)
    X_train = np.where(np.isinf(X_train), np.nan, X_train)
    X_test = np.where(np.isinf(X_test), np.nan, X_test)

    imputer = SimpleImputer(strategy="median")
    X_train = imputer.fit_transform(X_train)
    X_test = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc = scaler.transform(X_test)

    # B0 baselines
    b0_dir = np.random.randint(0, 2, size=len(y_test_dir))
    b0_ret = np.full(len(y_test_ret), y_train_ret.mean())

    clf_results["B0 Random"]["fold_acc"].append(accuracy_score(y_test_dir, b0_dir))
    clf_results["B0 Random"]["preds"].extend(b0_dir)
    clf_results["B0 Random"]["true"].extend(y_test_dir)

    reg_results["B0 Mean"]["fold_mae"].append(mean_absolute_error(y_test_ret, b0_ret))
    reg_results["B0 Mean"]["fold_rmse"].append(np.sqrt(mean_squared_error(y_test_ret, b0_ret)))
    reg_results["B0 Mean"]["preds"].extend(b0_ret)
    reg_results["B0 Mean"]["true"].extend(y_test_ret)

    # classification
    for name, model, scaled in CLF_MODELS:
        print(f"  Training {name} (clf)...")
        Xtr, Xte = (X_train_sc, X_test_sc) if scaled else (X_train, X_test)
        model.fit(Xtr, y_train_dir)
        preds = model.predict(Xte)
        probs = model.predict_proba(Xte)[:, 1]

        clf_results[name]["fold_acc"].append(accuracy_score(y_test_dir, preds))
        clf_results[name]["preds"].extend(preds)
        clf_results[name]["probs"].extend(probs)
        clf_results[name]["true"].extend(y_test_dir)

    # regression
    for name, model, scaled in REG_MODELS:
        print(f"  Training {name} (reg)...")
        Xtr, Xte = (X_train_sc, X_test_sc) if scaled else (X_train, X_test)
        model.fit(Xtr, y_train_ret)
        preds = model.predict(Xte)

        reg_results[name]["fold_mae"].append(mean_absolute_error(y_test_ret, preds))
        reg_results[name]["fold_rmse"].append(np.sqrt(mean_squared_error(y_test_ret, preds)))
        reg_results[name]["preds"].extend(preds)
        reg_results[name]["true"].extend(y_test_ret)

print("\nDone.")

Walk-forward: 5 folds (sliding 7-year window)
------------------------------------------------------------

Fold 1: train [2014-2020] (13,169) -> test [2021] (1,970)
  Training Logistic Regression (clf)...
  Training Random Forest (clf)...
  Training XGBoost (clf)...
  Training Linear Regression (reg)...
  Training Random Forest (reg)...
  Training XGBoost (reg)...

Fold 2: train [2015-2021] (13,313) -> test [2022] (1,975)
  Training Logistic Regression (clf)...
  Training Random Forest (clf)...
  Training XGBoost (clf)...
  Training Linear Regression (reg)...
  Training Random Forest (reg)...
  Training XGBoost (reg)...

Fold 3: train [2016-2022] (13,439) -> test [2023] (1,984)
  Training Logistic Regression (clf)...
  Training Random Forest (clf)...
  Training XGBoost (clf)...
  Training Linear Regression (reg)...
  Training Random Forest (reg)...
  Training XGBoost (reg)...

Fold 4: train [2017-2023] (13,560) -> test [2024] (2,000)
  Training Logistic Regression (clf)...
  Training 

In [15]:
# Results
print("=" * 75)
print("=== Direction (Classification) — Walk-Forward ===")
print(f"{'Model':<25} {'Avg DA%':>10} {'Pooled F1':>10} {'Prec':>8} {'Recall':>8} {'AUC':>8}")
print("-" * 75)

for name in ["B0 Random"] + [n for n, _, _ in CLF_MODELS]:
    r = clf_results[name]
    true = np.array(r["true"])
    preds = np.array(r["preds"])

    avg_acc = np.mean(r["fold_acc"])
    f1 = f1_score(true, preds, average="weighted")
    prec = precision_score(true, preds, average="weighted")
    rec = recall_score(true, preds, average="weighted")

    if r["probs"]:
        auc = roc_auc_score(true, np.array(r["probs"]))
        auc_str = f"{auc:>8.4f}"
    else:
        auc_str = f"{'—':>8}"

    print(f"{name:<25} {avg_acc:>10.4f} {f1:>10.4f} {prec:>8.4f} {rec:>8.4f} {auc_str}")

print(f"\n=== Magnitude (Regression) — Walk-Forward ===")
print(f"{'Model':<25} {'Avg MAE':>10} {'Avg RMSE':>10} {'Pooled R²':>10}")
print("-" * 60)

for name in ["B0 Mean"] + [n for n, _, _ in REG_MODELS]:
    r = reg_results[name]
    true = np.array(r["true"])
    preds = np.array(r["preds"])

    avg_mae = np.mean(r["fold_mae"])
    avg_rmse = np.mean(r["fold_rmse"])
    r2 = r2_score(true, preds)

    print(f"{name:<25} {avg_mae:>10.4f} {avg_rmse:>10.4f} {r2:>10.4f}")

=== Direction (Classification) — Walk-Forward ===
Model                        Avg DA%  Pooled F1     Prec   Recall      AUC
---------------------------------------------------------------------------
B0 Random                     0.5005     0.5027   0.5097   0.5005        —
Logistic Regression           0.5171     0.5189   0.5231   0.5170   0.5162
Random Forest                 0.5410     0.4926   0.5086   0.5409   0.5078
XGBoost                       0.5222     0.5134   0.5114   0.5222   0.5034

=== Magnitude (Regression) — Walk-Forward ===
Model                        Avg MAE   Avg RMSE  Pooled R²
------------------------------------------------------------
B0 Mean                       0.0419     0.0576    -0.0020
Linear Regression             0.0451     0.1194    -6.7741
Random Forest                 0.0426     0.0590    -0.0574
XGBoost                       0.0463     0.0632    -0.2132


In [16]:
# Save models (last fold, trained on 2018-2024) and results
os.makedirs("models", exist_ok=True)

for name, model, _ in CLF_MODELS:
    path = f"models/b1_{name.lower().replace(' ', '_')}_clf.pkl"
    joblib.dump(model, path)
    print(f"Saved {path}")

for name, model, _ in REG_MODELS:
    path = f"models/b1_{name.lower().replace(' ', '_')}_reg.pkl"
    joblib.dump(model, path)
    print(f"Saved {path}")

with open("models/b1_results.pkl", "wb") as f:
    pickle.dump({"clf_results": clf_results, "reg_results": reg_results}, f)
    print("Saved models/b1_results.pkl")

Saved models/b1_logistic_regression_clf.pkl
Saved models/b1_random_forest_clf.pkl
Saved models/b1_xgboost_clf.pkl
Saved models/b1_linear_regression_reg.pkl
Saved models/b1_random_forest_reg.pkl
Saved models/b1_xgboost_reg.pkl
Saved models/b1_results.pkl
